# RAG Operations Notebook

Interactive notebook for managing RAG knowledge-base collections directly through `src/rag/ops` and Qdrant.
This notebook does **not** depend on `experiments/rag/notebook_ops.py`.

## Operator Flow
1. Setup & Direct Imports
2. Knowledge Bases & Current State
3. Initial Bootstrap Build
4. Update Existing Alias From `_meta`
5. Alias Management
6. Diagnostics
7. Direct Qdrant Exploration
8. Direct Qdrant Management

## Newcomer Notes
- Alias names use `{kb}_{role}`, for example `arxiv_champion`.
- Temporary rebuild aliases use `{kb}_{role}_staging`.
- Physical collections use `{kb}_{timestamp}`.
- The **first** collection build is done manually from this notebook.
- Airflow DAGs later refresh existing aliases by reading `_meta` from the current target collection.
- Read-only cells are safe to run. Write cells require setting an explicit action variable first.

## 1. Setup & Direct Imports

In [25]:
from pathlib import Path
from pprint import pprint

from rag.ops import (
    BuildConfig,
    ImplementationInfo,
    assign_alias_to_collection,
    create_arxiv_collection,
    create_pytorch_docs_collection,
    detach_alias,
    inspect_alias,
    inspect_collection,
    list_alias_mappings,
    promote_alias,
    update_arxiv_collection,
    update_pytorch_docs_collection,
)
from rag.vector_store import QdrantVectorStore
from shared.config import bootstrap_local_settings_env, get_knowledge_bases, get_settings


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current


repo_root = find_repo_root()
loaded_env = bootstrap_local_settings_env(repo_root=repo_root)
settings = get_settings()

ARXIV_DATA_FILE = repo_root / "assets" / "rag_data" / "arxiv" / "arxiv_papers.json"
PYTORCH_DOCS_FILE = repo_root / "assets" / "rag_data" / "pytorch_docs" / "pytorch_docs.json"

print(f"Repo root: {repo_root}")
print(f"Loaded env: {loaded_env}")
print(f"Qdrant endpoint: {settings.qdrant_host}:{settings.qdrant_port}")
print(f"ArXiv data file exists: {ARXIV_DATA_FILE.exists()} -> {ARXIV_DATA_FILE}")
print(f"PyTorch docs data file exists: {PYTORCH_DOCS_FILE.exists()} -> {PYTORCH_DOCS_FILE}")

Repo root: /home/jovyan
Loaded env: None
Qdrant endpoint: qdrant:6333
ArXiv data file exists: True -> /home/jovyan/assets/rag_data/arxiv/arxiv_papers.json
PyTorch docs data file exists: True -> /home/jovyan/assets/rag_data/pytorch_docs/pytorch_docs.json


## 2. Knowledge Bases & Current State

Start here before changing anything. It shows which knowledge bases exist, which aliases are valid for each one, and what aliases currently resolve in Qdrant.

In [21]:
kb_registry = {
    task_name: [
        {
            "name": kb.name,
            "aliases": kb.aliases,
            "update_strategy": kb.update_strategy,
            "label": kb.label,
        }
        for kb in task_cfg.knowledge_bases
    ]
    for task_name, task_cfg in get_knowledge_bases().items()
}

pprint({"knowledge_bases": kb_registry})
print("\nLive aliases:")
pprint(list_alias_mappings())

{'knowledge_bases': {'chat': [{'aliases': ['champion', 'challenger'],
                               'label': 'ArXiv papers (ML / AI theory)',
                               'name': 'arxiv',
                               'update_strategy': 'incremental'}],
                     'code': [{'aliases': ['champion', 'challenger'],
                               'label': 'PyTorch docs',
                               'name': 'pytorch_docs',
                               'update_strategy': 'replace'}]}}

Live aliases:
[{'alias_name': 'arxiv_champion', 'collection_name': 'arxiv_20260401_140632'},
 {'alias_name': 'pytorch_docs_champion_staging',
  'collection_name': 'pytorch_docs_20260401_140541'}]


## 3. Initial Bootstrap Build

Use this section when the environment is empty and no `champion` or `challenger` aliases exist yet.
This is the first build path for a new service deployment. It creates a fresh collection, writes `_meta`, and can optionally attach an initial alias.

In [ ]:
INITIAL_BUILD_ACTION = "create_pytorch_docs"  # "create_arxiv" or "create_pytorch_docs"
INITIAL_ALIAS = "champion"  # "champion", "challenger", or None
INITIAL_COLLECTION_NAME = None
INITIAL_EMBEDDING_MODEL = None

ARXIV_CHUNKING_STRATEGY = "fixed_token"
PYTORCH_CHUNKING_STRATEGY = "code"
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64

# For hybrid search experiments, set:
#   SPARSE_ENCODER = "bm25"
#   RETRIEVAL_STRATEGY = "hybrid"
# Dense-only (default):
SPARSE_ENCODER = None
RETRIEVAL_STRATEGY = "dense"

if INITIAL_BUILD_ACTION == "create_arxiv":
    result = create_arxiv_collection(
        build_config=BuildConfig(
            chunking_strategy=ARXIV_CHUNKING_STRATEGY,
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            embedding_model=INITIAL_EMBEDDING_MODEL or settings.embedding_model,
            sparse_encoder=SPARSE_ENCODER,
            retrieval_strategy=RETRIEVAL_STRATEGY,
        ),
        arxiv_file=str(ARXIV_DATA_FILE),
        kb="arxiv",
        alias=INITIAL_ALIAS,
        collection_name=INITIAL_COLLECTION_NAME,
        implementation=ImplementationInfo(module="rag.ops.create.arxiv", experimental=False),
    )
    pprint(result)
elif INITIAL_BUILD_ACTION == "create_pytorch_docs":
    result = create_pytorch_docs_collection(
        build_config=BuildConfig(
            chunking_strategy=PYTORCH_CHUNKING_STRATEGY,
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            embedding_model=INITIAL_EMBEDDING_MODEL or settings.embedding_model,
            sparse_encoder=SPARSE_ENCODER,
            retrieval_strategy=RETRIEVAL_STRATEGY,
        ),
        pytorch_docs_file=str(PYTORCH_DOCS_FILE),
        kb="pytorch_docs",
        alias=INITIAL_ALIAS,
        collection_name=INITIAL_COLLECTION_NAME,
        implementation=ImplementationInfo(
            module="rag.ops.create.pytorch_docs",
            experimental=False,
        ),
    )
    pprint(result)
else:
    print(
        "Set INITIAL_BUILD_ACTION to 'create_arxiv' or 'create_pytorch_docs' to bootstrap "
        "the first collection."
    )

{'alias': {'alias_name': 'pytorch_docs_champion',
           'collection_name': 'pytorch_docs_20260402_074835',
           'meta': {'build_config': {'chunk_overlap': 64,
                                     'chunk_size': 512,
                                     'chunking_strategy': 'code',
                                     'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'},
                    'created_at': '2026-04-02T07:48:35.243785+00:00',
                    'implementation': {'experimental': False,
                                       'module': 'rag.ops.create.pytorch_docs'},
                    'kb_name': 'pytorch_docs'}},
 'collection_name': 'pytorch_docs_20260402_074835',
 'meta': {'build_config': {'chunk_overlap': 64,
                           'chunk_size': 512,
                           'chunking_strategy': 'code',
                           'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'},
          'created_at': '2026-04-02T07:48:35.243785+00:0

## 4. Update Existing Alias From `_meta`

Use this section after the first collection already exists and has valid `_meta`.
This is the same style of refresh that the DAGs use later: they target an existing alias and reconstruct the build from stored metadata.

**Legacy collections** created before the `sparse_encoder` / `retrieval_strategy` fields were added to `BuildConfig` cannot be refreshed in-place. They require a one-time rebuild via Section 3 so that `_meta.build_config` satisfies the new contract.

In [ ]:
UPDATE_ACTION = None  # "refresh_arxiv" or "refresh_pytorch_docs"
UPDATE_ALIAS = "champion"

if UPDATE_ACTION == "refresh_arxiv":
    pprint(
        update_arxiv_collection(
            arxiv_file=str(ARXIV_DATA_FILE),
            kb="arxiv",
            alias=UPDATE_ALIAS,
        )
    )
elif UPDATE_ACTION == "refresh_pytorch_docs":
    pprint(
        update_pytorch_docs_collection(
            pytorch_docs_file=str(PYTORCH_DOCS_FILE),
            kb="pytorch_docs",
            alias=UPDATE_ALIAS,
        )
    )
else:
    print(
        "Set UPDATE_ACTION to 'refresh_arxiv' or 'refresh_pytorch_docs' to run the "
        "_meta-driven refresh path."
    )

## 5. Alias Management

Use these production-safe alias operations after collections already exist.
This is where you attach `champion` or `challenger`, promote one alias to another, or detach an alias entirely.

In [ ]:
KB = "arxiv"

ALIAS_ACTION = None  # "assign", "promote", or "detach"
COLLECTION_NAME = "arxiv_YYYYMMDD_HHMMSS"  # Used by "assign"
ASSIGN_ALIAS = "challenger"
FROM_ALIAS = "challenger"
TO_ALIAS = "champion"
DETACH_ALIAS = "challenger"

if ALIAS_ACTION == "assign":
    pprint(
        assign_alias_to_collection(
            kb=KB,
            alias=ASSIGN_ALIAS,
            collection_name=COLLECTION_NAME,
        )
    )
elif ALIAS_ACTION == "promote":
    pprint(promote_alias(kb=KB, from_alias=FROM_ALIAS, to_alias=TO_ALIAS))
elif ALIAS_ACTION == "detach":
    pprint(detach_alias(kb=KB, alias=DETACH_ALIAS))
else:
    print("Set ALIAS_ACTION to one of: assign, promote, detach")

## 6. Diagnostics

Use the strict `rag.ops` inspection helpers first.
If strict inspection fails because `_meta` is missing or malformed, jump to Section 7.3 to read raw `_meta` directly from Qdrant.

In [22]:
DIAGNOSTIC_ACTION = "inspect_alias"  # "inspect_alias" or "inspect_collection"
KB = "arxiv"
ALIAS = "champion"
COLLECTION_NAME = "arxiv_YYYYMMDD_HHMMSS"

try:
    if DIAGNOSTIC_ACTION == "inspect_alias":
        pprint(inspect_alias(kb_name=KB, alias=ALIAS))
    elif DIAGNOSTIC_ACTION == "inspect_collection":
        pprint(inspect_collection(collection_name=COLLECTION_NAME))
    else:
        print("Set DIAGNOSTIC_ACTION to 'inspect_alias' or 'inspect_collection'.")
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")
    print("If strict inspection fails, use Section 7.3 to read raw _meta.")

ValueError: arxiv_20260401_140632.build_config: 'chunking_strategy' must be a non-empty string
If strict inspection fails, use Section 7.3 to read raw _meta.


## 7. Direct Qdrant Exploration

These cells talk to Qdrant directly instead of going through `rag.ops`.
Use them when you want raw state, need to understand how aliases resolve, or need to diagnose why the higher-level helpers fail.

Start with the connection cell below, then use the read-only examples in order.

In [23]:
admin_store = QdrantVectorStore(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
    collection_name="_qdrant_admin",
)

print(f"Connected to Qdrant at {settings.qdrant_host}:{settings.qdrant_port}")

Connected to Qdrant at qdrant:6333


### 7.1 List Collections And Aliases

This is the fastest way to see what exists in Qdrant right now.
It shows physical collections and alias mappings separately.

In [24]:
collection_names = sorted(
    collection.name for collection in admin_store.client.get_collections().collections
)
alias_rows = sorted(admin_store.list_aliases(), key=lambda row: row["alias_name"])

print(f"Collections: {len(collection_names)}")
pprint(collection_names)
print(f"\nAliases: {len(alias_rows)}")
pprint(alias_rows)

Collections: 2
['arxiv_20260401_140632', 'pytorch_docs_20260401_140541']

Aliases: 2
[{'alias_name': 'arxiv_champion', 'collection_name': 'arxiv_20260401_140632'},
 {'alias_name': 'pytorch_docs_champion_staging',
  'collection_name': 'pytorch_docs_20260401_140541'}]


### 7.2 Resolve An Alias Or Inspect A Collection

This answers the common question: "what does this name actually point to?"
Set `LOOKUP_NAME` to either an alias name like `arxiv_champion` or a concrete collection name like `arxiv_20260401_140632`.

In [19]:
LOOKUP_NAME = "arxiv_champion"  # Alias or collection name

resolved_collection = admin_store.resolve_alias(LOOKUP_NAME)
target_name = resolved_collection or LOOKUP_NAME
target_store = QdrantVectorStore(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
    collection_name=target_name,
)

pprint(
    {
        "lookup_name": LOOKUP_NAME,
        "resolved_collection": resolved_collection,
        "collection_info": target_store.get_collection_info(),
    }
)

{'collection_info': {'exists': True, 'points_count': 1},
 'lookup_name': 'arxiv_champion',
 'resolved_collection': 'arxiv_20260401_140632'}


### 7.3 Read Raw `_meta`

Use this when `inspect_alias()` or `inspect_collection()` fails.
It reads the raw metadata sentinel without strict schema validation, which is useful for legacy or partially rebuilt collections.

In [13]:
TARGET_NAME = "arxiv_champion"  # Alias or collection name

resolved_collection = admin_store.resolve_alias(TARGET_NAME)
meta_target = resolved_collection or TARGET_NAME
meta_store = QdrantVectorStore(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
    collection_name=meta_target,
)

pprint(
    {
        "lookup_name": TARGET_NAME,
        "resolved_collection": resolved_collection,
        "raw_meta": meta_store.read_meta(),
    }
)

{'lookup_name': 'arxiv_champion',
 'raw_meta': {'build_config': {'chunk_overlap': None,
                               'chunk_size': None,
                               'chunking_strategy': None,
                               'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'},
              'created_at': '2026-04-01T14:06:32.965590+00:00',
              'kb_name': 'arxiv',
              'type': 'collection_meta'},
 'resolved_collection': 'arxiv_20260401_140632'}


### 7.4 Find Collections Without Aliases

This helps you spot orphan collections that are no longer reachable through an alias.
It is useful before cleanup and after failed rebuilds.

In [14]:
all_collections = {
    collection.name for collection in admin_store.client.get_collections().collections
}
aliased_collections = {row["collection_name"] for row in admin_store.list_aliases()}
orphan_collections = sorted(all_collections - aliased_collections)

pprint(
    {
        "orphan_collections": orphan_collections,
        "aliased_collections": sorted(aliased_collections),
    }
)

{'aliased_collections': ['arxiv_20260401_140632',
                         'pytorch_docs_20260401_140541'],
 'orphan_collections': []}


## 8. Direct Qdrant Management

These cells can modify Qdrant directly and bypass the production-safe validation in `rag.ops`.
Use them for manual repair or cleanup only, and double-check names before you run them.

Nothing happens until you set an explicit action value.

In [ ]:
RAW_ALIAS_ACTION = None  # "update" or "delete"
RAW_ALIAS_NAME = "pytorch_docs_champion_staging"
RAW_COLLECTION_NAME = "pytorch_docs_YYYYMMDD_HHMMSS"  # Used by "update"

if RAW_ALIAS_ACTION == "update":
    admin_store.update_alias(alias_name=RAW_ALIAS_NAME, collection_name=RAW_COLLECTION_NAME)
    pprint({"alias_name": RAW_ALIAS_NAME, "collection_name": RAW_COLLECTION_NAME})
elif RAW_ALIAS_ACTION == "delete":
    admin_store.delete_alias(alias_name=RAW_ALIAS_NAME)
    print(f"Deleted alias: {RAW_ALIAS_NAME}")
else:
    print("Set RAW_ALIAS_ACTION to 'update' or 'delete' to modify a raw alias.")

In [ ]:
DELETE_COLLECTION_NAME = None
CONFIRM_DELETE = False

if DELETE_COLLECTION_NAME:
    preview_store = QdrantVectorStore(
        host=settings.qdrant_host,
        port=settings.qdrant_port,
        collection_name=DELETE_COLLECTION_NAME,
    )
    pprint(
        {
            "collection_name": DELETE_COLLECTION_NAME,
            "collection_info": preview_store.get_collection_info(),
            "raw_meta": preview_store.read_meta(),
        }
    )

if DELETE_COLLECTION_NAME and CONFIRM_DELETE:
    admin_store.delete_collection(DELETE_COLLECTION_NAME)
    print(f"Deleted collection: {DELETE_COLLECTION_NAME}")
else:
    print(
        "Set DELETE_COLLECTION_NAME and CONFIRM_DELETE = True "
        "to delete a collection after previewing it."
    )